# 10: Watch noise change a quantum state

**Level:** Intermediate  
**Before you start:** Notebooks 01 and 09.  
**Resources:** CPU unless an optional remote step is enabled.

Can you separate physical noise from the randomness of measurement?

Run each cell in order. All core calculations are written in this notebook.

## 1. Prepare the excited state

An X gate prepares |1⟩. Amplitude damping describes relaxation from |1⟩ toward |0⟩. We will vary its probability and use exact density-matrix evolution, with no shot sampling.

In [ ]:
import flagquantum as fq
import flagquantum.noise as fqn
import numpy as np
import matplotlib.pyplot as plt

q = fq.Circuit(1).x(0)
noise = fqn.NoiseModel().add("x", fqn.amplitude_damping_channel(0.2))
r = fq.run(
    q,
    noise_model=noise,
    options=fq.ExecutionOptions(mode="density_matrix"),
)
rho = r.state[0]
print("Density matrix:", rho)
print("Z expectation:", (rho[0, 0] - rho[1, 1]).real.item())


## 2. Predict the curve

If damping probability is p, P(1)=1-p and ⟨Z⟩=2p-1. At p=0 the answer is -1; at p=1 it is +1.

The density matrix diagonal gives P(0) and P(1). We read it directly and compute Z = P(0) - P(1); the current generic Pauli-output path does not handle this density-matrix result.


In [ ]:
probabilities = np.linspace(0, 1, 11)
observed = []
for p in probabilities:
    model = fqn.NoiseModel().add("x", fqn.amplitude_damping_channel(float(p)))
    result = fq.run(
        q,
        noise_model=model,
        options=fq.ExecutionOptions(mode="density_matrix"),
    )
    rho = result.state[0]
    observed.append((rho[0, 0] - rho[1, 1]).real.item())
np.testing.assert_allclose(observed, 2 * probabilities - 1, atol=1e-6)
plt.plot(probabilities, 2 * probabilities - 1, label="Theory")
plt.scatter(probabilities, observed, label="FlagQuantum")
plt.xlabel("Damping probability")
plt.ylabel("Z expectation")
plt.legend()
plt.show()


## Make it yours

Prepare |0⟩ instead, or apply noise after a Hadamard gate. Work out the expected answer first. Compare this smooth physical change with shot-to-shot variation in notebook 09. A noise model is a modeling choice, not automatically a calibrated description of a real device.